In [20]:
import joblib
from dataclasses import dataclass, field
from typing import Tuple, Dict, Any

## load model

In [21]:
@dataclass(frozen=True)
class FinalCfg:
    test_size: float = 0.2
    random_state: int = 42
    oof_splits: int = 5
    weight_win: float = 2.0
    weight_loss: float = 1.0
    drop_non_features: Tuple[str, ...] = ('game_id','home_team','away_team','season','week')
    categorical_cols: Tuple[str, ...] = ('roof','surface')
    boolean_cols: Tuple[str, ...] = (
        'is_playoff','is_final_week','home_qb_switch','away_qb_switch','is_home_qb_new','is_away_qb_new'
    )
    # For OOF weighting model
    base_xgb_params: Dict[str, Any] = field(default_factory=lambda: dict(
        n_estimators=500, max_depth=3, learning_rate=0.01, min_child_weight=3,
        subsample=0.6, colsample_bytree=0.6, reg_alpha=1.0, reg_lambda=3.0,
        objective='reg:squarederror', random_state=42, tree_method='hist', n_jobs=1
    ))

In [22]:
res = joblib.load('/content/drive/MyDrive/BettingEdgeContinued/fantasy_model.pkl')

In [23]:
print(res.keys())
# → dict_keys(['mae', 'r2', 'y_test', 'y_pred', 'pipeline', 'used_columns', 'config'])

dict_keys(['mae', 'r2', 'y_test', 'y_pred', 'pipeline', 'used_columns', 'config'])


In [24]:
# The pipeline is what actually makes predictions
pipeline = res["pipeline"]
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['roof', 'surface']),
                                                 ('num', 'passthrough',
                                                  ['spread_line', 'away_rest',
                                                   'home_rest', 'total_line',
                                                   'div_game', 'temp', 'wind',
                                                   'home_rolling_avg_epa',
                                                   'home_rolling_avg_yards',
                                                   'home_rolling_play_count',
                                                   'away_rolling_avg_e...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.01,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=2, max_leaves=None, min_child_weight=3,
                              missing=nan, monotone_constraints=None,
                              multi_strategy=None, n_estimators=384, n_jobs=1,
                              num_parallel_tree=None, ...))])

In [25]:
# This tells you EXACTLY what columns the model was trained on
pre = pipeline.named_steps["preprocessor"]
pre

ColumnTransformer(transformers=[('cat',
                                 OrdinalEncoder(handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['roof', 'surface']),
                                ('num', 'passthrough',
                                 ['spread_line', 'away_rest', 'home_rest',
                                  'total_line', 'div_game', 'temp', 'wind',
                                  'home_rolling_avg_epa',
                                  'home_rolling_avg_yards',
                                  'home_rolling_play_count',
                                  'away_rolling_avg_epa',
                                  'away_rolling_avg_yards',
                                  'aw...
                                  'epa_home_def_away_off_rolling_diff',
                                  'avg_yards_home_off_away_def_rolling_diff',
                                  'avg_yards_home_def_away_off_rolling_diff',
                                  'play_count_home_off_away_def_rolling_diff',
                                  'play_count_home_def_away_off_rolling_diff',
                                  'home_recent_sos_opponent_avg',
                                  'home_season_sos_opponent_avg',
                                  'away_recent_sos_opponent_avg',
                                  'away_season_sos_opponent_avg', 'sos_diff', ...])],
                  verbose_feature_names_out=False)

In [26]:
cat_cols  = pre.transformers_[0][2]   # OrdinalEncoder columns (roof, surface)
num_cols  = pre.transformers_[1][2]   # passthrough columns (everything else)

In [27]:
print("Categorical features:", cat_cols)
print("Numeric/Boolean features:", num_cols)
print("\nTotal features expected:", len(cat_cols) + len(num_cols))

Categorical features: ['roof', 'surface']
Numeric/Boolean features: ['spread_line', 'away_rest', 'home_rest', 'total_line', 'div_game', 'temp', 'wind', 'home_rolling_avg_epa', 'home_rolling_avg_yards', 'home_rolling_play_count', 'away_rolling_avg_epa', 'away_rolling_avg_yards', 'away_rolling_play_count', 'home_rolling_allowed_avg_epa', 'home_rolling_allowed_avg_yards', 'home_rolling_allowed_play_count', 'away_rolling_allowed_avg_epa', 'away_rolling_allowed_avg_yards', 'away_rolling_allowed_play_count', 'epa_home_off_away_def_rolling_diff', 'epa_home_def_away_off_rolling_diff', 'avg_yards_home_off_away_def_rolling_diff', 'avg_yards_home_def_away_off_rolling_diff', 'play_count_home_off_away_def_rolling_diff', 'play_count_home_def_away_off_rolling_diff', 'home_recent_sos_opponent_avg', 'home_season_sos_opponent_avg', 'away_recent_sos_opponent_avg', 'away_season_sos_opponent_avg', 'sos_diff', 'season_sos_diff', 'home_allpro_last_3_years_weighted', 'away_allpro_last_3_years_weighted', 'diff

In [28]:
import nflreadpy as nfl
import pandas as pd

raw = nfl.load_schedules([2025])

print(type(raw))   # confirms it's a polars DataFrame

# Convert Polars → Pandas correctly
schedule = raw.to_pandas()

print(schedule.shape)
print(schedule.columns.tolist())

<class 'polars.dataframe.frame.DataFrame'>
(285, 46)
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


## simple feature wrangling

In [29]:
key_cols = [
    'game_id', 'home_team', 'away_team', 'week', 'season',
    'spread_line', 'total_line', 'away_rest', 'home_rest',
    'div_game', 'roof', 'surface', 'temp', 'wind', 'result'
]

for col in key_cols:
    status = "✅" if col in schedule.columns else "❌ MISSING"
    print(f"{status}  {col}")

print("\n\nAll available columns:")
print(schedule.columns.tolist())

✅  game_id
✅  home_team
✅  away_team
✅  week
✅  season
✅  spread_line
✅  total_line
✅  away_rest
✅  home_rest
✅  div_game
✅  roof
✅  surface
✅  temp
✅  wind
✅  result


All available columns:
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


In [30]:
TARGET_WEEK = 10  # change this each week you want to predict

week_games = schedule[schedule['week'] == TARGET_WEEK]

# Show just the columns relevant to us
cols = ['game_id', 'home_team', 'away_team', 'gameday', 'gametime',
        'spread_line', 'total_line', 'away_rest', 'home_rest',
        'div_game', 'roof', 'surface', 'temp', 'wind']

print(week_games[cols].to_string())

             game_id home_team away_team     gameday gametime  spread_line  total_line  away_rest  home_rest  div_game      roof     surface  temp  wind
135   2025_10_LV_DEN       DEN        LV  2025-11-06    20:15          9.5        42.5          4          4         1  outdoors       grass  60.0  10.0
136  2025_10_ATL_IND       IND       ATL  2025-11-09    09:30          6.5        48.5          7          7         0    closed       grass  46.0   2.0
137   2025_10_NO_CAR       CAR        NO  2025-11-09    13:00          5.5        38.5          7          7         1  outdoors       grass  73.0  15.0
138  2025_10_NYG_CHI       CHI       NYG  2025-11-09    13:00          4.5        45.5          7          7         0  outdoors       grass  33.0  10.0
139  2025_10_JAX_HOU       HOU       JAX  2025-11-09    13:00         -1.5        37.5          7          7         1    closed   astroturf   NaN   NaN
140  2025_10_BUF_MIA       MIA       BUF  2025-11-09    13:00         -8.5        

In [31]:
# All completed games BEFORE the target week — used to build rolling features
history = schedule[
    (schedule['week'] < TARGET_WEEK) &
    (schedule['result'].notna())   # result is NaN for games not yet played
].copy()

# The games you actually want to predict
upcoming = schedule[schedule['week'] == TARGET_WEEK].copy()

# Dome games have no temp/wind — fill with season medians from history
upcoming['temp'] = upcoming['temp'].fillna(72)
upcoming['wind'] = upcoming['wind'].fillna(0)

print(f"History: {len(history)} completed games (weeks 1 through {TARGET_WEEK - 1})")
print(f"Upcoming: {len(upcoming)} games to predict in week {TARGET_WEEK}")

History: 135 completed games (weeks 1 through 9)
Upcoming: 14 games to predict in week 10


In [32]:
# These are the Group 1 features the model expects
group1_features = [
    'spread_line', 'away_rest', 'home_rest', 'total_line',
    'div_game', 'temp', 'wind', 'roof', 'surface'
]

print("Group 1 feature check for upcoming games:\n")
print(upcoming[group1_features].to_string())
print("\nNull counts:")
print(upcoming[group1_features].isnull().sum())

Group 1 feature check for upcoming games:

     spread_line  away_rest  home_rest  total_line  div_game  temp  wind      roof     surface
135          9.5          4          4        42.5         1  60.0  10.0  outdoors       grass
136          6.5          7          7        48.5         0  46.0   2.0    closed       grass
137          5.5          7          7        38.5         1  73.0  15.0  outdoors       grass
138          4.5          7          7        45.5         0  33.0  10.0  outdoors       grass
139         -1.5          7          7        37.5         1  72.0   0.0    closed   astroturf
140         -8.5          7         10        50.5         1  84.0   6.0  outdoors       grass
141         -4.5         10          7        48.5         0  72.0   0.0      dome   sportturf
142         -1.5         14         14        37.5         0  62.0  12.0  outdoors   fieldturf
143          2.5          7         14        48.5         0  82.0  11.0  outdoors       grass
144    

## pbp feature wrangling

In [33]:
raw_pbp = nfl.load_pbp([2025])
pbp = raw_pbp.to_pandas()

print(pbp.shape)
print(pbp['play_type'].value_counts())

(48771, 372)
play_type
pass           19735
run            14893
no_play         4732
kickoff         2918
punt            2042
extra_point     1324
field_goal      1140
qb_kneel         452
qb_spike          80
Name: count, dtype: int64


In [34]:
pbp_rp = pbp[
    pbp['play_type'].isin(['run', 'pass']) &
    pbp['posteam'].notna() &
    pbp['defteam'].notna()
].copy()

print(pbp_rp.shape)
print(pbp_rp[['game_id', 'posteam', 'defteam', 'epa', 'yards_gained', 'play_type']].head(10))

(34628, 372)
           game_id posteam defteam       epa  yards_gained play_type
2   2025_01_ARI_NO     ARI      NO -0.190052           3.0       run
3   2025_01_ARI_NO     ARI      NO  1.317340          11.0      pass
4   2025_01_ARI_NO     ARI      NO -1.694360         -11.0      pass
5   2025_01_ARI_NO     ARI      NO -1.284150          -2.0       run
6   2025_01_ARI_NO     ARI      NO -0.840574           1.0       run
8   2025_01_ARI_NO      NO     ARI -0.194728           3.0       run
9   2025_01_ARI_NO      NO     ARI -0.788527           0.0      pass
10  2025_01_ARI_NO      NO     ARI -1.545796           0.0      pass
12  2025_01_ARI_NO     ARI      NO  0.041066           5.0      pass
13  2025_01_ARI_NO     ARI      NO  1.133550          13.0       run


In [35]:
# Offense stats — grouped by game and the team with the ball
off_stats = (
    pbp_rp
    .groupby(['game_id', 'posteam'])
    .agg(
        avg_epa=('epa', 'mean'),
        avg_yards=('yards_gained', 'mean'),
        play_count=('play_id', 'count')
    )
    .reset_index()
    .rename(columns={'posteam': 'team'})
)

# Defense stats — grouped by game and the team defending
def_stats = (
    pbp_rp
    .groupby(['game_id', 'defteam'])
    .agg(
        allowed_avg_epa=('epa', 'mean'),
        allowed_avg_yards=('yards_gained', 'mean'),
        allowed_play_count=('play_id', 'count')
    )
    .reset_index()
    .rename(columns={'defteam': 'team'})
)

print("Offense stats shape:", off_stats.shape)
print(off_stats.head())

Offense stats shape: (570, 5)
           game_id team   avg_epa  avg_yards  play_count
0   2025_01_ARI_NO  ARI  0.043989   4.524590          61
1   2025_01_ARI_NO   NO -0.041654   4.701493          67
2  2025_01_BAL_BUF  BAL  0.385699   8.640000          50
3  2025_01_BAL_BUF  BUF  0.234987   6.519481          77
4  2025_01_CAR_JAX  CAR -0.256380   4.266667          60


In [36]:
# Pull just the week/season info from the schedule to attach to pbp stats
week_lookup = schedule[['game_id', 'week', 'season']].drop_duplicates()

off_stats = off_stats.merge(week_lookup, on='game_id', how='left')
def_stats = def_stats.merge(week_lookup, on='game_id', how='left')

print(off_stats[['game_id', 'team', 'week', 'avg_epa']].head(10))

           game_id team  week   avg_epa
0   2025_01_ARI_NO  ARI     1  0.043989
1   2025_01_ARI_NO   NO     1 -0.041654
2  2025_01_BAL_BUF  BAL     1  0.385699
3  2025_01_BAL_BUF  BUF     1  0.234987
4  2025_01_CAR_JAX  CAR     1 -0.256380
5  2025_01_CAR_JAX  JAX     1  0.152458
6  2025_01_CIN_CLE  CIN     1 -0.108868
7  2025_01_CIN_CLE  CLE     1 -0.051202
8  2025_01_DAL_PHI  DAL     1  0.035278
9  2025_01_DAL_PHI  PHI     1  0.160450


In [37]:
# Sort by team and week so the rolling window goes in the right direction
off_stats = off_stats.sort_values(['team', 'season', 'week']).reset_index(drop=True)
def_stats = def_stats.sort_values(['team', 'season', 'week']).reset_index(drop=True)

# Rolling offense — shift(1) means "not including the current game"
# This is critical — you can't use a game's own stats to predict that game
for feat in ['avg_epa', 'avg_yards', 'play_count']:
    off_stats[f'rolling_{feat}'] = (
        off_stats
        .groupby('team')[feat]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    )

# Rolling defense
for feat in ['allowed_avg_epa', 'allowed_avg_yards', 'allowed_play_count']:
    def_stats[f'rolling_{feat}'] = (
        def_stats
        .groupby('team')[feat]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    )

print("Offense rolling stats:")
print(off_stats[['team', 'week', 'avg_epa', 'rolling_avg_epa']].head(15))

Offense rolling stats:
   team  week   avg_epa  rolling_avg_epa
0   ARI     1  0.043989              NaN
1   ARI     2  0.127011         0.043989
2   ARI     3 -0.011040         0.085500
3   ARI     4 -0.081304         0.053320
4   ARI     5 -0.074265         0.019664
5   ARI     6  0.089873         0.000878
6   ARI     7 -0.052211         0.010055
7   ARI     9  0.113321        -0.025789
8   ARI    10 -0.210903        -0.000917
9   ARI    11  0.061463        -0.026837
10  ARI    12 -0.146952         0.000309
11  ARI    13 -0.026304        -0.047056
12  ARI    14 -0.061863        -0.041875
13  ARI    15  0.012079        -0.076912
14  ARI    16 -0.004240        -0.032315


In [38]:
# We only need the rolling columns, not the raw game-level stats
off_rolling = off_stats[['game_id', 'team', 'rolling_avg_epa', 'rolling_avg_yards', 'rolling_play_count']]
def_rolling = def_stats[['game_id', 'team', 'rolling_allowed_avg_epa', 'rolling_allowed_avg_yards', 'rolling_allowed_play_count']]

# Merge home team offense
upcoming = upcoming.merge(
    off_rolling.rename(columns={'team': 'home_team', **{c: f'home_{c}' for c in off_rolling.columns if c.startswith('rolling')}}),
    on=['game_id', 'home_team'], how='left'
)

# Merge away team offense
upcoming = upcoming.merge(
    off_rolling.rename(columns={'team': 'away_team', **{c: f'away_{c}' for c in off_rolling.columns if c.startswith('rolling')}}),
    on=['game_id', 'away_team'], how='left'
)

# Merge home team defense
upcoming = upcoming.merge(
    def_rolling.rename(columns={'team': 'home_team', **{c: f'home_{c}' for c in def_rolling.columns if c.startswith('rolling')}}),
    on=['game_id', 'home_team'], how='left'
)

# Merge away team defense
upcoming = upcoming.merge(
    def_rolling.rename(columns={'team': 'away_team', **{c: f'away_{c}' for c in def_rolling.columns if c.startswith('rolling')}}),
    on=['game_id', 'away_team'], how='left'
)

print(upcoming[['home_team', 'away_team', 'home_rolling_avg_epa', 'away_rolling_avg_epa',
                 'home_rolling_allowed_avg_epa', 'away_rolling_allowed_avg_epa']].to_string())

   home_team away_team  home_rolling_avg_epa  away_rolling_avg_epa  home_rolling_allowed_avg_epa  away_rolling_allowed_avg_epa
0        DEN        LV              0.069126             -0.223643                     -0.077516                      0.046081
1        IND       ATL              0.223933             -0.010268                     -0.047286                      0.040593
2        CAR        NO              0.001531             -0.216599                      0.081096                      0.015716
3        CHI       NYG              0.123259              0.068929                     -0.022078                      0.164295
4        HOU       JAX              0.063374              0.016247                     -0.210792                      0.086189
5        MIA       BUF             -0.085888              0.138325                      0.011105                     -0.054874
6        MIN       BAL             -0.090772             -0.060594                      0.133357               

In [39]:
# These were computed in your notebook as combinations of offense vs opponent defense
upcoming['epa_home_off_away_def_rolling_diff'] = (
    upcoming['home_rolling_avg_epa'] - upcoming['away_rolling_allowed_avg_epa']
)
upcoming['epa_home_def_away_off_rolling_diff'] = (
    upcoming['home_rolling_allowed_avg_epa'] - upcoming['away_rolling_avg_epa']
)
upcoming['avg_yards_home_off_away_def_rolling_diff'] = (
    upcoming['home_rolling_avg_yards'] - upcoming['away_rolling_allowed_avg_yards']
)
upcoming['avg_yards_home_def_away_off_rolling_diff'] = (
    upcoming['home_rolling_allowed_avg_yards'] - upcoming['away_rolling_avg_yards']
)
upcoming['play_count_home_off_away_def_rolling_diff'] = (
    upcoming['home_rolling_play_count'] - upcoming['away_rolling_allowed_play_count']
)
upcoming['play_count_home_def_away_off_rolling_diff'] = (
    upcoming['home_rolling_allowed_play_count'] - upcoming['away_rolling_play_count']
)

# Sanity check
diff_cols = [c for c in upcoming.columns if 'diff' in c]
print(upcoming[['home_team', 'away_team'] + diff_cols].to_string())

   home_team away_team  epa_home_off_away_def_rolling_diff  epa_home_def_away_off_rolling_diff  avg_yards_home_off_away_def_rolling_diff  avg_yards_home_def_away_off_rolling_diff  play_count_home_off_away_def_rolling_diff  play_count_home_def_away_off_rolling_diff
0        DEN        LV                            0.023046                            0.146127                                  0.574017                                 -0.620338                                       -2.4                                       12.0
1        IND       ATL                            0.183339                           -0.037018                                  1.095865                                 -0.690890                                       -0.4                                        9.4
2        CAR        NO                           -0.014186                            0.297695                                  0.049364                                  0.192464                           

In [40]:
# How many of the 79 features do I have so far?
model_features = cat_cols + num_cols   # from step 1 earlier

have = [f for f in model_features if f in upcoming.columns]
missing = [f for f in model_features if f not in upcoming.columns]

print(f"Have: {len(have)}/79")
print(f"\nStill missing ({len(missing)}):")
for f in missing:
    print(f"  - {f}")

Have: 27/79

Still missing (52):
  - home_recent_sos_opponent_avg
  - home_season_sos_opponent_avg
  - away_recent_sos_opponent_avg
  - away_season_sos_opponent_avg
  - sos_diff
  - season_sos_diff
  - home_allpro_last_3_years_weighted
  - away_allpro_last_3_years_weighted
  - diff_allpro_last_3_years_weighted
  - home_allpro_prev_year
  - away_allpro_prev_year
  - diff_allpro_prev_year
  - home_offense_allpro_3_years
  - away_offense_allpro_3_years
  - home_defense_allpro_3_years
  - away_defense_allpro_3_years
  - allpro_diff_home_off_away_def_3_years
  - allpro_diff_home_def_away_off_3_years 
  - home_offense_allpro_prev_year
  - away_offense_allpro_prev_year
  - home_defense_allpro_prev_year
  - away_defense_allpro_prev_year
  - allpro_diff_home_off_away_def_prev_year
  - allpro_diff_home_def_away_off_prev_year
  - league_rolling_avg_abs_margin_by_week
  - home_qbr_prev_year
  - away_qbr_prev_year
  - diff_qbr_prev_year
  - home_injured_count
  - away_injured_count
  - diff_injured

## Historic Feature Wrangling

In [41]:
# Split each game into two rows — one for home team, one for away team
home_games = history[['season', 'week', 'home_team', 'away_team', 'home_score', 'away_score']].copy()
home_games.columns = ['season', 'week', 'team', 'opponent', 'team_score', 'opp_score']

away_games = history[['season', 'week', 'away_team', 'home_team', 'away_score', 'home_score']].copy()
away_games.columns = ['season', 'week', 'team', 'opponent', 'team_score', 'opp_score']

long_df = pd.concat([home_games, away_games], ignore_index=True)
long_df = long_df.sort_values(['team', 'season', 'week']).reset_index(drop=True)

# Mark wins
long_df['team_win'] = (long_df['team_score'] > long_df['opp_score']).astype(int)

print(long_df.head(10))
print(long_df.shape)

   season  week team opponent  team_score  opp_score  team_win
0    2025     1  ARI       NO          20         13         1
1    2025     2  ARI      CAR          27         22         1
2    2025     3  ARI       SF          15         16         0
3    2025     4  ARI      SEA          20         23         0
4    2025     5  ARI      TEN          21         22         0
5    2025     6  ARI      IND          27         31         0
6    2025     7  ARI       GB          23         27         0
7    2025     9  ARI      DAL          27         17         1
8    2025     1  ATL       TB          20         23         0
9    2025     2  ATL      MIN          22          6         1
(270, 7)


In [42]:
# shift() again — same reason as rolling PBP stats
# A team's win % entering a game can't include that game itself
long_df['win_pct'] = (
    long_df
    .groupby('team')['team_win']
    .transform(lambda x: x.shift(1).expanding().mean())
)

print(long_df[['team', 'week', 'team_win', 'win_pct']].head(20))

   team  week  team_win   win_pct
0   ARI     1         1       NaN
1   ARI     2         1  1.000000
2   ARI     3         0  1.000000
3   ARI     4         0  0.666667
4   ARI     5         0  0.500000
5   ARI     6         0  0.400000
6   ARI     7         0  0.333333
7   ARI     9         1  0.285714
8   ATL     1         0       NaN
9   ATL     2         1  0.000000
10  ATL     3         0  0.500000
11  ATL     4         1  0.333333
12  ATL     6         1  0.500000
13  ATL     7         0  0.600000
14  ATL     8         0  0.500000
15  ATL     9         0  0.428571
16  BAL     1         0       NaN
17  BAL     2         1  0.000000
18  BAL     3         0  0.500000
19  BAL     4         0  0.333333


In [43]:
# Build a lookup: for each team/week, what is their win %?
win_pct_lookup = long_df[['season', 'week', 'team', 'win_pct']].copy()
win_pct_lookup.columns = ['season', 'week', 'opponent', 'opponent_win_pct']

# Merge opponent's win % onto each row
long_df = long_df.merge(win_pct_lookup, on=['season', 'week', 'opponent'], how='left')

print(long_df[['team', 'week', 'opponent', 'win_pct', 'opponent_win_pct']].head(20))

   team  week opponent   win_pct  opponent_win_pct
0   ARI     1       NO       NaN               NaN
1   ARI     2      CAR  1.000000          0.000000
2   ARI     3       SF  1.000000          1.000000
3   ARI     4      SEA  0.666667          0.666667
4   ARI     5      TEN  0.500000          0.000000
5   ARI     6      IND  0.400000          0.800000
6   ARI     7       GB  0.333333          0.600000
7   ARI     9      DAL  0.285714          0.375000
8   ATL     1       TB       NaN               NaN
9   ATL     2      MIN  0.000000          1.000000
10  ATL     3      CAR  0.500000          0.000000
11  ATL     4      WAS  0.333333          0.666667
12  ATL     6      BUF  0.500000          0.800000
13  ATL     7       SF  0.600000          0.666667
14  ATL     8      MIA  0.500000          0.142857
15  ATL     9       NE  0.428571          0.750000
16  BAL     1      BUF       NaN               NaN
17  BAL     2      CLE  0.000000          0.000000
18  BAL     3      DET  0.50000

In [44]:
# Recent SOS = avg win % of last 3 opponents
long_df['recent_sos'] = (
    long_df
    .groupby('team')['opponent_win_pct']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean().fillna(0))
)

# Season SOS = avg win % of all opponents so far this season
long_df['season_sos'] = (
    long_df
    .groupby('team')['opponent_win_pct']
    .transform(lambda x: x.shift(1).expanding().mean().fillna(0))
)

print(long_df[['team', 'week', 'recent_sos', 'season_sos']].head(20))

   team  week  recent_sos  season_sos
0   ARI     1    0.000000    0.000000
1   ARI     2    0.000000    0.000000
2   ARI     3    0.000000    0.000000
3   ARI     4    0.500000    0.500000
4   ARI     5    0.555556    0.555556
5   ARI     6    0.555556    0.416667
6   ARI     7    0.488889    0.493333
7   ARI     9    0.466667    0.511111
8   ATL     1    0.000000    0.000000
9   ATL     2    0.000000    0.000000
10  ATL     3    1.000000    1.000000
11  ATL     4    0.500000    0.500000
12  ATL     6    0.555556    0.555556
13  ATL     7    0.488889    0.616667
14  ATL     8    0.711111    0.626667
15  ATL     9    0.536508    0.546032
16  BAL     1    0.000000    0.000000
17  BAL     2    0.000000    0.000000
18  BAL     3    0.000000    0.000000
19  BAL     4    0.250000    0.250000


In [45]:
sos_lookup = long_df[['season', 'week', 'team', 'recent_sos', 'season_sos']].copy()

# We need the SOS for the TARGET week — meaning what was each team's SOS entering that week
# Since long_df only has history, we need the last known value per team
latest_sos = (
    sos_lookup
    .sort_values(['team', 'season', 'week'])
    .groupby('team')
    .last()
    .reset_index()
    [['team', 'recent_sos', 'season_sos']]
)

# Merge for home team
upcoming = upcoming.merge(
    latest_sos.rename(columns={
        'team': 'home_team',
        'recent_sos': 'home_recent_sos_opponent_avg',
        'season_sos': 'home_season_sos_opponent_avg'
    }),
    on='home_team', how='left'
)

# Merge for away team
upcoming = upcoming.merge(
    latest_sos.rename(columns={
        'team': 'away_team',
        'recent_sos': 'away_recent_sos_opponent_avg',
        'season_sos': 'away_season_sos_opponent_avg'
    }),
    on='away_team', how='left'
)

# Fill any NaN with 0 (same as notebook)
for col in ['home_recent_sos_opponent_avg', 'home_season_sos_opponent_avg',
            'away_recent_sos_opponent_avg', 'away_season_sos_opponent_avg']:
    upcoming[col] = upcoming[col].fillna(0)

print(upcoming[['home_team', 'away_team',
                'home_recent_sos_opponent_avg', 'away_recent_sos_opponent_avg',
                'home_season_sos_opponent_avg', 'away_season_sos_opponent_avg']].to_string())

   home_team away_team  home_recent_sos_opponent_avg  away_recent_sos_opponent_avg  home_season_sos_opponent_avg  away_season_sos_opponent_avg
0        DEN        LV                      0.253968                      0.483333                      0.632653                      0.547222
1        IND       ATL                      0.403175                      0.536508                      0.446599                      0.546032
2        CAR        NO                      0.355556                      0.638095                      0.450000                      0.666327
3        CHI       NYG                      0.311111                      0.726984                      0.294444                      0.454422
4        HOU       JAX                      0.543651                      0.588889                      0.521825                      0.627778
5        MIA       BUF                      0.422222                      0.523810                      0.359524                      0.261905

In [46]:
upcoming['sos_diff'] = (
    upcoming['home_recent_sos_opponent_avg'] - upcoming['away_recent_sos_opponent_avg']
)
upcoming['season_sos_diff'] = (
    upcoming['home_season_sos_opponent_avg'] - upcoming['away_season_sos_opponent_avg']
)

print(upcoming[['home_team', 'away_team', 'sos_diff', 'season_sos_diff']].to_string())

   home_team away_team  sos_diff  season_sos_diff
0        DEN        LV -0.229365         0.085431
1        IND       ATL -0.133333        -0.099433
2        CAR        NO -0.282540        -0.216327
3        CHI       NYG -0.415873        -0.159977
4        HOU       JAX -0.045238        -0.105952
5        MIA       BUF -0.101587         0.097619
6        MIN       BAL -0.009524         0.134127
7        NYJ       CLE -0.022222         0.044444
8         TB        NE  0.521429         0.181009
9        SEA       ARI  0.183333         0.091667
10        SF        LA  0.038889        -0.009921
11       WAS       DET -0.109524         0.141950
12       LAC       PIT  0.111111         0.110317
13        GB       PHI  0.033333        -0.094444


In [47]:
have = [f for f in model_features if f in upcoming.columns]
missing = [f for f in model_features if f not in upcoming.columns]

print(f"Have: {len(have)}/79")
print(f"\nStill missing ({len(missing)}):")
for f in missing:
    print(f"  - {f}")

Have: 33/79

Still missing (46):
  - home_allpro_last_3_years_weighted
  - away_allpro_last_3_years_weighted
  - diff_allpro_last_3_years_weighted
  - home_allpro_prev_year
  - away_allpro_prev_year
  - diff_allpro_prev_year
  - home_offense_allpro_3_years
  - away_offense_allpro_3_years
  - home_defense_allpro_3_years
  - away_defense_allpro_3_years
  - allpro_diff_home_off_away_def_3_years
  - allpro_diff_home_def_away_off_3_years 
  - home_offense_allpro_prev_year
  - away_offense_allpro_prev_year
  - home_defense_allpro_prev_year
  - away_defense_allpro_prev_year
  - allpro_diff_home_off_away_def_prev_year
  - allpro_diff_home_def_away_off_prev_year
  - league_rolling_avg_abs_margin_by_week
  - home_qbr_prev_year
  - away_qbr_prev_year
  - diff_qbr_prev_year
  - home_injured_count
  - away_injured_count
  - diff_injured_count
  - diff_active_allpro_weighted
  - diff_active_allpro_prev_year
  - home_rolling_win_pct
  - away_rolling_win_pct
  - sack_diff
  - sack_diff_reverse
  - tur